# Do the institution-type gaps differ in London?

`a-level-institution-type.ipynb` compares academy converters, sponsor-led academies, free schools / UTCs / studio schools and LA-maintained schools, and independent schools and colleges, in an England-wide model with region and local-authority effects. It assumes a type gap (converter against sponsor-led, say) is the **same in every region**. London is different in mix (43% of its state schools with GCSE results are converters against 55-71% elsewhere; 26% are LA maintained; 13% are free schools, UTCs or studio schools) and in level (the highest region by a wide margin at GCSE), so this notebook tests the assumption.

It refits the same model with **a London-specific extra shift for each state type**, in general GCSE quality and, for each of the seven subject groups, in A-level value added beyond the GCSE profile. The main type effects in this fit describe institutions outside London; London's are the main effect plus the extra. Region and local-authority effects are in the model as before, so what is tested is whether the *gap between types* is different in London, not whether London is higher (it is, and that is in the regional effects).

London has 389 schools with GCSE results (about 170 converters, 75 sponsor-led, 100 maintained and 50 free / UTC / studio), so its gaps are estimated less precisely than the rest of England's, and an interval that includes zero for the difference means "not shown", not "the same". One year of results; association only.

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt
import xarray as xr

%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")
print(f"Running on PyMC v{pm.__version__}")

## Data

In [ ]:
raw = pd.read_csv("data/all-value-add-errors.csv")
names = pd.read_csv("data/school-names.csv").set_index("URN")

# ---- GCSE: six elements per school with known standard errors (schools with Progress 8 results)
elements = ["English", "Maths", "Science", "Humanities", "Languages", "Open"]
column = {"English": "P8MEAENG", "Maths": "P8MEAMAT", "Science": "SCIVAMEA_PTQ_EE",
          "Humanities": "HUMVAMEA_PTQ_EE", "Languages": "LANVAMEA_PTQ_EE", "Open": "P8MEAOPEN"}
frames = []
for e in elements:
    c = column[e]
    sub = raw[["URN", c, f"{c} lower", f"{c} upper"]].dropna()
    sub.columns = ["URN", "va", "lower", "upper"]
    sub["element"] = e
    frames.append(sub)
long = pd.concat(frames)
long["se"] = (long["upper"] - long["lower"]) / (2 * 1.96)
n_el = long.groupby("URN")["element"].nunique()
long = long[long["URN"].isin(n_el[n_el >= 3].index)].sort_values("URN").reset_index(drop=True)

# ---- all institutions: schools with GCSE results first, then institutions with A-level results only
urns_gcse = pd.Index(sorted(long["URN"].unique()))
urns_only = pd.Index(sorted(set(raw["URN"]) - set(urns_gcse)))
inst = urns_gcse.append(urns_only)
n_g, n_inst = len(urns_gcse), len(inst)
no_gcse = (np.arange(n_inst) >= n_g).astype(float)

long["school_idx"] = urns_gcse.get_indexer(long["URN"])
long["element_idx"] = long["element"].map({e: k for k, e in enumerate(elements)}).to_numpy()
n_elements = len(elements)
x_obs, x_se = long["va"].to_numpy(), long["se"].to_numpy()
s_idx, e_idx = long["school_idx"].to_numpy(), long["element_idx"].to_numpy()

region_series = raw.drop_duplicates("URN").set_index("URN")["RGN24NM"].reindex(inst)
regions = list(region_series.value_counts().index)
reg_idx = region_series.map({r: k for k, r in enumerate(regions)}).fillna(len(regions)).astype(int).to_numpy()   # unknown region -> national average
is_london = (region_series.to_numpy() == "London")
print(f"{n_g} schools with GCSE results, {len(urns_only)} institutions with A-level results only, {n_inst} in all; {is_london.sum()} in London")


# ---- local authority for every institution
la_series = names["local_authority"].reindex(inst)
la_names = sorted(la_series.unique())
la_idx = la_series.map({a: k for k, a in enumerate(la_names)}).to_numpy()

# ---- institution type, from the public register's detailed establishment type
def type_category(t):
    if t in ("Other independent school", "Other independent special school"):
        return "independent (fee-paying)"
    if t in ("Academy converter", "Academy special converter"):
        return "academy converter"
    if t in ("Academy sponsor led", "Academy special sponsor led"):
        return "academy sponsor-led"
    if t in ("Free schools", "Free schools special", "University technical college", "Studio schools", "City technology college", "Free schools alternative provision"):
        return "free school / UTC / studio"
    if t in ("Community school", "Voluntary aided school", "Voluntary controlled school", "Foundation school"):
        return "LA maintained"
    if t in ("Further education", "Sixth form centres", "Academy 16-19 converter", "Academy 16 to 19 sponsor led", "Free schools 16 to 19"):
        return "college (post-16)"
    return "other"
type_names = ["independent (fee-paying)", "academy converter", "academy sponsor-led", "free school / UTC / studio", "LA maintained", "college (post-16)", "other"]
state4 = type_names[1:5]                      # state-funded types that have GCSE results
other3 = [type_names[0], type_names[5], type_names[6]]
inst_type = np.array([type_names.index(type_category(t)) for t in names["type_detail"].reindex(inst).to_numpy()])
is4 = ((inst_type >= 1) & (inst_type <= 4)).astype(float)
t4_idx = np.where(is4 == 1, inst_type - 1, 0)
is3 = 1.0 - is4
t3_idx = np.array([{0: 0, 5: 1, 6: 2}.get(k, 0) for k in inst_type])

# years as an academy (converters and sponsor-led), in decades, centred on a typical 12 years
open_year = pd.to_datetime(names["open_date"].reindex(inst), format="%d-%m-%Y", errors="coerce").dt.year.to_numpy()
is_cs = ((inst_type == 1) | (inst_type == 2)).astype(float)
cs_idx = np.where(inst_type == 2, 1, 0)      # 0 converter, 1 sponsor-led
years_c = is_cs * ((2024 - np.nan_to_num(open_year, nan=2012)) / 10.0 - 1.2)
summary = pd.crosstab(pd.Series(np.array(type_names)[inst_type], name="type"), pd.Series(np.where(no_gcse == 1, "A-level results only", "has GCSE results"), name=""))
print(summary.reindex(type_names).fillna(0).astype(int).to_string())

In [ ]:
arts = ["Art & Design", "Art & Design (Fine Art)", "Art & Design (Photography)", "Art & Design (Graphics)", "Art & Design (Textiles)",
        "Art & Design (3d Studies)", "Art & Design (Critical Studies)", "Music", "Music Technology", "Drama & Theatre Studies", "Dance"]
groups = {"Maths": ["Mathematics"],
          "Sciences": ["Biology", "Chemistry", "Physics"],
          "English": ["English Literature", "English Language", "English Language & Literature"],
          "Humanities": ["History", "Geography", "Religious Studies", "Logic/ Philosophy", "Ancient History", "Classical Civilisation"],
          "Social sciences": ["Psychology", "Sociology", "Economics", "Government & Politics", "Law"],
          "Business & Computing": ["Business Studies:Single", "Computer Studies/Computing"],
          "Creative arts": arts}
group_names = list(groups)
n_groups = len(group_names)

raw_i = raw.set_index("URN").reindex(inst)
frames = []
for gname, subjects in groups.items():
    va = pd.DataFrame({s: raw_i[f"A-level {s} VA"] for s in subjects})
    se = pd.DataFrame({s: (raw_i[f"A-level {s} VA upper"] - raw_i[f"A-level {s} VA lower"]) / (2 * 1.96) for s in subjects})
    ent = pd.DataFrame({s: raw_i[f"A-level {s} entries"].where(va[s].notna(), 0).fillna(0) for s in subjects})
    total = ent.sum(axis=1)
    pooled_va = (va.fillna(0) * ent).sum(axis=1) / total.replace(0, np.nan)
    pooled_se = np.sqrt(((se.fillna(0) * ent) ** 2).sum(axis=1)) / total.replace(0, np.nan)   # entry-weighted; assumes separate cohorts
    f = pd.DataFrame({"inst_idx": np.arange(n_inst), "group": gname, "va": pooled_va.to_numpy(), "se": pooled_se.to_numpy(), "entries": total.to_numpy()})
    frames.append(f.dropna(subset=["va", "se"]))
alevel = pd.concat(frames).reset_index(drop=True)
alevel["group_idx"] = alevel["group"].map({g: k for k, g in enumerate(group_names)}).to_numpy()
y_obs, y_se = alevel["va"].to_numpy(), alevel["se"].to_numpy()
ys_idx, yg_idx = alevel["inst_idx"].to_numpy(), alevel["group_idx"].to_numpy()
assert (y_se > 0).all()
print(f"{len(alevel)} institution-group A-level observations")

## Model

The model of `a-level-institution-type.ipynb`, with the London interaction switched on (`interact=True`).

In [ ]:
keep = np.array([0.0 if e == "Humanities" else 1.0 for e in elements])

def build_model(interact=False):
    """interact=True adds London-specific type effects (extra shifts for London institutions, by type)."""
    coords = {"element": elements, "region": regions, "group": group_names, "la": la_names, "type4": state4, "type3": other3,
              "cs": ["academy converter", "academy sponsor-led"]}
    with pm.Model(coords=coords) as model:
        # ---- geography and institution type: where general GCSE quality sits
        sigma_m = pm.HalfNormal("sigma_m", 0.5)
        m = pm.ZeroSumNormal("m", sigma=sigma_m, dims="region")
        m_all = pt.concatenate([m, pt.zeros(1)])
        sigma_a = pm.HalfNormal("sigma_a", 0.3)
        a_la = pm.Normal("a_la", 0, sigma_a, dims="la")
        type_g = pm.ZeroSumNormal("type_g", sigma=0.5, dims="type4")       # type shift in general GCSE quality (state types with GCSE results)
        type_h = pm.ZeroSumNormal("type_h", sigma=0.5, dims="type4")       # type shift in the tilt
        type_c = pm.ZeroSumNormal("type_c", sigma=0.3, dims="type4")       # type shift in log consistency
        slope_g = pm.Normal("slope_g", 0, 0.3, dims="cs")                  # change in general quality per decade as an academy
        g_loc = m_all[reg_idx] + a_la[la_idx] + is4 * type_g[t4_idx] + is_cs * years_c * slope_g[cs_idx]
        if interact:
            ix_g = pm.ZeroSumNormal("ix_g", sigma=0.3, dims="type4")            # extra type shift in general GCSE quality for London
            g_loc = g_loc + is_london.astype(float) * is4 * ix_g[t4_idx]
        h_loc = is4 * type_h[t4_idx]

        # ---- GCSE side, schools with GCSE results only
        mu = pm.Normal("mu", 0, 1, dims="element")
        lam = pm.HalfNormal("lam", 1, dims="element")
        tau = pm.HalfNormal("tau", 0.5, dims="element")
        g = pm.Normal("g", g_loc[:n_g], 1, shape=n_g)
        h = pm.Normal("h", h_loc[:n_g], 1, shape=n_g)
        k_raw = pm.Normal("kappa_raw", 0, 0.5, shape=n_elements)
        kappa = pm.Deterministic("kappa", k_raw * keep, dims="element")   # Humanities fixed at 0; the sign of the tilt is fixed after sampling
        sigma_s = pm.HalfNormal("sigma_s", 0.5)
        rho = pm.Deterministic("rho", 2 * pm.Beta("rho_raw", 2, 2) - 1)
        w = pm.Normal("w", 0, 1, shape=n_g)
        log_s = pm.Deterministic("log_s", is4[:n_g] * type_c[t4_idx[:n_g]] + sigma_s * (rho * (g - g_loc[:n_g]) + pt.sqrt(1 - rho**2) * w))
        pm.Normal("x_obs", mu=mu[e_idx] + lam[e_idx] * g[s_idx] + kappa[e_idx] * h[s_idx],
                  sigma=pt.sqrt((tau[e_idx] * pt.exp(log_s[s_idx]))**2 + x_se**2), observed=x_obs)

        # institutions without GCSE results: the typical GCSE quality for where they are and what they are
        g_full = pt.concatenate([g, g_loc[n_g:]])
        h_full = pt.concatenate([h, h_loc[n_g:]])

        # ---- A-level side (all institutions)
        nu = pm.Normal("nu", 0, 0.5, dims="group")
        sd_group = pm.HalfNormal("sd_group", 0.5, dims="group")
        sd_group_ng = pm.HalfNormal("sd_group_ng", 0.5, dims="group")
        sigma_psi = pm.HalfNormal("sigma_psi", 0.3)
        psi = pm.ZeroSumNormal("psi", sigma=sigma_psi, dims="region")
        psi_all = pt.concatenate([psi, pt.zeros(1)])
        sigma_xi = pm.HalfNormal("sigma_xi", 0.2)
        xi = pm.Normal("xi", 0, sigma_xi, dims="la")
        slope_u = pm.Normal("slope_u", 0, 0.3, dims="cs")                  # change in shared A-level quality per decade as an academy
        u = pm.Normal("u", 0, 1, shape=n_inst)
        b = pm.Normal("b", 0, 0.5, dims="group")
        c = pm.Normal("c", 0, 0.5, dims="group")
        lam_a = pm.HalfNormal("lam_a", 0.5, dims="group")
        type_a = pm.ZeroSumNormal("type_a", sigma=0.3, dims=("group", "type4"))   # type shift at A-level beyond GCSE, state types, by subject group
        d3 = pm.Normal("d3", 0, 0.5, dims=("group", "type3"))                      # mean shift for independent schools, colleges and others, by group
        ix_term = 0.0
        if interact:
            ix_a = pm.ZeroSumNormal("ix_a", sigma=0.15, dims=("group", "type4"))   # extra type shift at A-level beyond GCSE for London
            ix_term = is_london[ys_idx].astype(float) * is4[ys_idx] * ix_a[yg_idx, t4_idx[ys_idx]]
        shared = psi_all[reg_idx] + xi[la_idx] + is_cs * years_c * slope_u[cs_idx] + u
        y_mean = (nu[yg_idx] + b[yg_idx] * g_full[ys_idx] + c[yg_idx] * h_full[ys_idx] + lam_a[yg_idx] * shared[ys_idx]
                  + is4[ys_idx] * type_a[yg_idx, t4_idx[ys_idx]] + is3[ys_idx] * d3[yg_idx, t3_idx[ys_idx]] + ix_term)
        sd_y = sd_group[yg_idx] * (1 - no_gcse[ys_idx]) + sd_group_ng[yg_idx] * no_gcse[ys_idx]
        pm.Normal("y_obs", mu=y_mean, sigma=pt.sqrt(sd_y**2 + y_se**2), observed=y_obs)
    return model

In [ ]:
model = build_model(interact=True)

### Fit

Four chains of 500 draws after 1,500 tuning steps, as in the main notebook.

In [ ]:
with model:
    idata = pm.sample(draws=500, tune=1500, chains=4, target_accept=0.99, random_seed=RANDOM_SEED, progressbar=False)

### Diagnostics

In [ ]:
n_div = int(idata.sample_stats["diverging"].sum())
keep_vars = ["type_g", "type_a", "ix_g", "ix_a", "sigma_m", "sigma_a", "sigma_psi", "sigma_xi", "d3", "nu", "b", "lam_a"]
ds = idata.posterior.to_dataset()[keep_vars]
rh, es = az.rhat(ds), az.ess(ds)
print(f"divergences = {n_div}")
display(pd.DataFrame({"max r_hat": {v: float(rh[v].max()) for v in rh.data_vars}, "min bulk ESS": {v: float(es[v].min()) for v in es.data_vars}}).sort_values("max r_hat", ascending=False).round(3))
def draws(name):
    a = ds[name].to_numpy(); return a.reshape(-1, *a.shape[2:])
tg, ta, ixg, ixa = draws("type_g"), draws("type_a"), draws("ix_g"), draws("ix_a")
import gc; del idata; gc.collect()

## The gaps outside London and in London

For each pair of state types, the gap outside London, the gap in London, and the difference between them (London minus outside), with 89% intervals. "General GCSE quality" is in SD units (about 0.45 GCSE value-added points per SD); the A-level rows are value-added points beyond the GCSE profile, by subject group. **P(London larger)** is the probability that the gap is bigger in London.

In [ ]:
def fmt(x, dp=2):
    lo, hi = np.percentile(x, [5.5, 94.5]); return f"{x.mean():+.{dp}f} [{lo:+.{dp}f}, {hi:+.{dp}f}]"

def london_gap(a_name, b_name):
    ia, ib = state4.index(a_name), state4.index(b_name)
    out_g = tg[:, ia] - tg[:, ib]
    lon_g = out_g + ixg[:, ia] - ixg[:, ib]
    rows = [{"": "general GCSE quality (SD)", "outside London": fmt(out_g), "London": fmt(lon_g), "London minus outside": fmt(lon_g - out_g), "P(London larger)": round(float((lon_g > out_g).mean()), 2)}]
    for j, gname in enumerate(group_names):
        out_a = ta[:, j, ia] - ta[:, j, ib]
        lon_a = out_a + ixa[:, j, ia] - ixa[:, j, ib]
        rows.append({"": f"A-level beyond GCSE: {gname}", "outside London": fmt(out_a), "London": fmt(lon_a), "London minus outside": fmt(lon_a - out_a), "P(London larger)": round(float((lon_a > out_a).mean()), 2)})
    return pd.DataFrame(rows).set_index("")

for a, b in [("academy converter", "academy sponsor-led"), ("academy converter", "LA maintained"), ("academy sponsor-led", "LA maintained")]:
    print(f"{a} minus {b}:")
    display(london_gap(a, b))

### The same, for the converter versus sponsor-led gap at a glance

The gap between converters and sponsor-led academies at A-level beyond GCSE, by subject group, outside London and in London.

In [ ]:
ia, ib = state4.index("academy converter"), state4.index("academy sponsor-led")
out_a = ta[:, :, ia] - ta[:, :, ib]
lon_a = out_a + ixa[:, :, ia] - ixa[:, :, ib]
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for arr, label, col, off in [(out_a, "outside London", "#4C72B0", -0.12), (lon_a, "London", "#DD8452", 0.12)]:
    lo, med, hi = np.percentile(arr, [5.5, 50, 94.5], axis=0)
    for j in range(n_groups):
        ax.plot([lo[j], hi[j]], [j + off, j + off], color=col, linewidth=2.5, label=label if j == 0 else None); ax.plot(med[j], j + off, "o", color=col)
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_yticks(range(n_groups), group_names); ax.invert_yaxis()
ax.set_xlabel("converter minus sponsor-led, A-level beyond GCSE (points)"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## Summary

**The gap between converters and sponsor-led academies at GCSE is about half as big in London.** In general GCSE quality it is $+0.77$ SD [0.65, 0.89] outside London (about 0.35 GCSE value-added points) and $+0.41$ [0.18, 0.63] in London (about 0.18 points); the difference is $-0.36$ [-0.61, -0.11], with probability 0.01 that the gap is larger in London. The two comparisons with LA-maintained schools point the same way but less firmly: converters minus maintained is $+0.42$ outside and $+0.22$ in London (difference $-0.20$ [-0.44, +0.03]), and sponsor-led minus maintained is $-0.35$ outside and $-0.19$ in London (difference $+0.15$ [-0.14, +0.43]). So in London the two academy routes differ less at GCSE, mainly because sponsor-led academies there sit closer to the others than they do elsewhere. This test cannot say why: London's mix of types is not unusual for sponsor-led academies (about 19% of its state schools, within the 12% to 28% range across regions), so how those academies were set up and supported would need to be looked at separately.

**At A-level, beyond the GCSE profile, the type gaps are not shown to differ in London.** For converter minus sponsor-led, London minus outside is between $-0.06$ and $+0.06$ points in every subject group, with every interval including zero. In London the gap is still positive: Maths $+0.14$ [0.05, 0.24], Humanities $+0.08$ [0.01, 0.15] and Business & Computing $+0.13$ [0.03, 0.23] are clear of zero; Sciences ($+0.07$ [-0.01, 0.15]), English, Social sciences ($0.00$) and Creative arts have intervals that include zero, which reflects how small London's sample is more than a different gap. Two of the 21 A-level differences have intervals that exclude zero, both for converters against maintained schools (Social sciences $-0.09$ [-0.16, -0.01] and Business & Computing $-0.11$ [-0.21, -0.01], London converters doing relatively worse). With 21 comparisons, two is about what chance produces, so we read nothing into them.

**Put together:** the assumption of the main notebook, that type gaps are the same everywhere, holds for the A-level gaps within the precision available, and does not hold for the GCSE gap between the two academy routes. Since the GCSE gap is smaller in London and the A-level gap beyond GCSE is about as large, the beyond-GCSE part is a larger share of the converter-versus-sponsor-led A-level gap in London than elsewhere.

**How much to trust it.** The fit has 5 divergences in 2,000 draws. The London-specific parameters mix well (effective sample sizes 419 for the A-level extras and 1,078 for the GCSE extra), but the type effects outside London are less well determined (ESS 77 for the GCSE type effects, about 300 for the A-level ones), and the spreads of the geographic effects did not converge ($\hat R$ up to 1.13, ESS 26-41). Those spreads are not used in the contrasts above, but they show that geography is only weakly identified in this fit, so treat the GCSE finding as one fit's evidence, worth confirming with a longer run. London has about 75 sponsor-led academies with GCSE results (against about 270 elsewhere), and the A-level intervals are wide accordingly. One year of results; association only.